In [95]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # add project root so src is importable

import pandas as pd
import numpy as np 
from src.config import TRANSACTIONS_RAW 

In [96]:
raw= pd.read_csv(TRANSACTIONS_RAW,skipinitialspace=True)
tran=raw.copy()
tran.columns=tran.columns.str.strip()
tran.head()

,customer_id,month,txn_count,txn_value_pkr
0,C100000,2024-07,6,9130.0
1,C100001,2024-07,15,33790.0
2,C100002,2024-07,4,6970.0
3,C100003,2024-07,14,21560.0
4,C100004,2024-07,4,5420.0


In [97]:
tran["month"].sample(10)

133084    Mar-2025
118144    2025-02 
5110      2024-07 
173585    2025-06 
13761     2024-07 
100020    Jan-2025
70329     Nov-2024
76674     2024-12 
167973    2025-06 
24709     2024-08 
Name: month, dtype: str

In [98]:
tran["month"]=tran["month"].str.strip()

In [99]:
tran["customer_id"]=tran["customer_id"].str.strip()

In [100]:
tran["month"].nunique()

12

In [101]:
tran.info()

<class 'pandas.DataFrame'>
RangeIndex: 178200 entries, 0 to 178199
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   customer_id    178200 non-null  str    
 1   month          178200 non-null  str    
 2   txn_count      178200 non-null  int64  
 3   txn_value_pkr  178200 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 7.9 MB


In [102]:
tran.describe().T

,count,mean,std,min,25%,50%,75%,max
txn_count,178200.0,7.090836,6.851674,-33.0,3.0,5.0,9.0,81.0
txn_value_pkr,178200.0,17369.921549,22213.721017,0.0,5110.0,10300.0,20770.0,577870.0


In [103]:
tran.isna().sum()

customer_id      0
month            0
txn_count        0
txn_value_pkr    0
dtype: int64

In [104]:
tran["month"].value_counts()

month
Nov-2024    14874
Mar-2025    14861
2025-02     14859
2025-06     14859
2024-07     14848
2024-09     14846
2024-12     14846
Jan-2025    14846
2025-04     14844
2024-10     14842
2025-05     14839
2024-08     14836
Name: count, dtype: int64

In [105]:
dupes= tran[tran.duplicated(["customer_id","month"],keep=False)]
dupes

,customer_id,month,txn_count,txn_value_pkr


In [106]:
tran["txn_count"]=tran["txn_count"].abs()

In [107]:
tran[tran["txn_count"] < 0]

,customer_id,month,txn_count,txn_value_pkr


In [108]:
dash= pd.to_datetime(tran["month"], format= "%Y-%m", errors="coerce")
word= pd.to_datetime(tran["month"], format= "%b-%Y", errors="coerce")

merge= dash.combine_first(word) 

In [109]:
tran["month"]=merge

In [110]:
tran["month"].head()

0   2024-07-01
1   2024-07-01
2   2024-07-01
3   2024-07-01
4   2024-07-01
Name: month, dtype: datetime64[us]

In [111]:
tran["month"].isna().sum() 

np.int64(0)

In [112]:
from src.config import AS_OF_DATE

REQUIRED = ["customer_id", "month", "txn_count", "txn_value_pkr"]

def validate_transactions(df):
    # composite key — one row per customer per month (this table's identity)
    assert df.duplicated(["customer_id", "month"]).sum() == 0, "duplicate (customer_id, month)"

    # join key must be clean — no stray whitespace, or the merge silently mismatches
    assert (df["customer_id"] == df["customer_id"].str.strip()).all(), "customer_id has whitespace"

    # counts and values non-negative (negatives were sign-fixed with abs; zero value allowed)
    assert (df["txn_count"] >= 0).all(), "negative txn_count"
    assert (df["txn_value_pkr"] >= 0).all(), "negative txn_value_pkr"

    # no month in the future
    assert (df["month"] <= AS_OF_DATE).all(), "month in the future"

    # completeness
    assert df[REQUIRED].notna().all().all(), "NaN in a required column"

    # NOTE: referential integrity (customer_id in customers) is deliberately NOT here —
    # it needs the merge, so it lives in the reconciliation file.
    return True